# Breast Cancer Classification with Scikit-learn\n\nThis notebook trains and evaluates a Random Forest classifier on the Wisconsin Diagnostic Breast Cancer dataset. **Educational use only; not for clinical diagnosis.**\n\n## Dataset source\nOriginal dataset: [UCI Machine Learning Repository — WDBC](https://archive.ics.uci.edu/dataset/17/breast+cancer+wisconsin+diagnostic). The data is loaded from scikit-learn's bundled `load_breast_cancer()` dataset, so no separate download is required.

In [ ]:
import pandas as pd\nimport matplotlib.pyplot as plt\nfrom sklearn.datasets import load_breast_cancer\nfrom sklearn.ensemble import RandomForestClassifier\nfrom sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,\n                             confusion_matrix, ConfusionMatrixDisplay, classification_report)\nfrom sklearn.model_selection import train_test_split

## 1. Load and inspect the dataset

In [ ]:
dataset = load_breast_cancer(as_frame=True)\nX = dataset.data\ny = dataset.target\ndf = X.copy()\ndf['target'] = y\nprint('Samples:', X.shape[0])\nprint('Features:', X.shape[1])\nprint('Target names:', dataset.target_names)\ndisplay(df.head())\ndisplay(y.map({i: name for i, name in enumerate(dataset.target_names)}).value_counts())

## 2. Create the train/test split\nWe reserve 20% of the data for testing. `stratify=y` preserves the class proportions in both sets, and `random_state=42` makes the split reproducible.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(\n    X, y, test_size=0.20, random_state=42, stratify=y\n)\nprint(f'Train: {X_train.shape[0]} samples')\nprint(f'Test:  {X_test.shape[0]} samples')

## 3. Train the model\nA Random Forest combines predictions from many decision trees. It is a reasonable baseline for tabular data and can model non-linear relationships.

In [ ]:
model = RandomForestClassifier(\n    n_estimators=200, random_state=42, class_weight='balanced'\n)\nmodel.fit(X_train, y_train)\ny_pred = model.predict(X_test)

## 4. Evaluate on the held-out test set\nAccuracy measures overall correctness. Precision, recall, and F1-score provide additional detail; the confusion matrix shows true versus predicted classes. The positive class for these binary metrics is target `1` (benign) in scikit-learn's dataset.

In [ ]:
metrics = {\n    'Accuracy': accuracy_score(y_test, y_pred),\n    'Precision (class 1: benign)': precision_score(y_test, y_pred),\n    'Recall (class 1: benign)': recall_score(y_test, y_pred),\n    'F1-score (class 1: benign)': f1_score(y_test, y_pred),\n}\npd.Series(metrics).round(4)

In [ ]:
cm = confusion_matrix(y_test, y_pred)\ndisplay(pd.DataFrame(cm, index=[f'True {n}' for n in dataset.target_names],\n                     columns=[f'Predicted {n}' for n in dataset.target_names]))\nConfusionMatrixDisplay(confusion_matrix=cm, display_labels=dataset.target_names).plot()\nplt.title('Random Forest — Test Set Confusion Matrix')\nplt.tight_layout()\nplt.show()\nprint(classification_report(y_test, y_pred, target_names=dataset.target_names))

## 5. Conclusion and limitations\nThe metrics above describe performance on a held-out portion of this benchmark dataset. Results do not establish clinical usefulness, and this model must not be used to diagnose or guide treatment. Further work could compare Logistic Regression and Support Vector Machines, use cross-validation, and inspect feature importance.